<a href="https://colab.research.google.com/github/fborrasumh/optimizia/blob/main/cuadernos/01-reforzamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# OptimizIA · Cuaderno 1: Formular y reforzar modelos enteros

**Técnicas Avanzadas de Optimización (8265)** · Máster en Estadística Computacional y Ciencia de Datos para la Toma de Decisiones · UMH.
Cuaderno de práctica de la app OptimizIA (Fernando Borrás Rocher, Mercedes Landete Ruiz y Juan Francisco Monge Ivars). Licencia MIT.

**Resultados de aprendizaje que trabaja:** RA1 (formular modelos de programación entera) y RA2 (reforzar una formulación y medir la mejora de la cota).

Idea central: dos formulaciones **correctas** del mismo problema entero pueden tener relajaciones lineales muy distintas. La más ajustada da mejores cotas y el branch-and-bound explora menos nodos.

> Cómo usarlo: ejecuta las celdas en orden (Mayús+Intro). Los ejercicios están marcados con ✏️. Las celdas «Solución» están plegadas: despliégalas solo después de intentarlo.
> Las instancias son **pequeñas y ficticias**, pensadas para ver el mecanismo; los tiempos que veas dependen de la máquina de Colab.

In [ ]:
# Instala OR-Tools si no está (en Colab tarda unos segundos)
try:
    import ortools
except ImportError:
    %pip install -q ortools
    import ortools
print('OR-Tools', ortools.__version__)

## 1. Localización de plantas sin capacidades (UFL)

Abrir la planta $i$ cuesta $f_i$; servir al cliente $j$ desde $i$ cuesta $c_{ij}$. Variables: $y_i\in\{0,1\}$ (abrir) y $x_{ij}\in[0,1]$ (fracción de $j$ servida desde $i$).

$$\min \sum_i f_i y_i + \sum_{i,j} c_{ij}x_{ij} \quad \text{s.a.}\quad \sum_i x_{ij}=1\;\forall j$$

Dos maneras de impedir servir desde una planta cerrada:
- **Débil (agregada):** $\sum_j x_{ij} \le n\,y_i \;\forall i$ — una restricción por planta.
- **Fuerte (desagregada):** $x_{ij} \le y_i \;\forall i,j$ — una por par.

Las dos tienen las mismas soluciones enteras. ¿Y sus relajaciones lineales?

In [ ]:
import random, time
from ortools.linear_solver import pywraplp

def instancia_ufl(m=8, n=25, semilla=1):
    rnd = random.Random(semilla)
    P = [(rnd.random(), rnd.random()) for _ in range(m)]   # plantas
    C = [(rnd.random(), rnd.random()) for _ in range(n)]   # clientes
    f = [round(rnd.uniform(3, 6), 2) for _ in range(m)]
    c = [[round(10 * ((P[i][0]-C[j][0])**2 + (P[i][1]-C[j][1])**2) ** .5, 2) for j in range(n)] for i in range(m)]
    return f, c

def ufl(f, c, fuerte=True, relajada=False):
    m, n = len(f), len(c[0])
    s = pywraplp.Solver.CreateSolver('GLOP' if relajada else 'SCIP')
    y = [s.NumVar(0, 1, f'y{i}') if relajada else s.BoolVar(f'y{i}') for i in range(m)]
    x = [[s.NumVar(0, 1, f'x{i}_{j}') for j in range(n)] for i in range(m)]
    for j in range(n):
        s.Add(sum(x[i][j] for i in range(m)) == 1)
    for i in range(m):
        if fuerte:
            for j in range(n): s.Add(x[i][j] <= y[i])
        else:
            s.Add(sum(x[i][j] for j in range(n)) <= n * y[i])
    s.Minimize(sum(f[i] * y[i] for i in range(m)) + sum(c[i][j] * x[i][j] for i in range(m) for j in range(n)))
    t0 = time.time(); st = s.Solve(); dt = time.time() - t0
    assert st == pywraplp.Solver.OPTIMAL
    return s.Objective().Value(), dt, s.NumConstraints(), s.nodes() if not relajada else 0

f, c = instancia_ufl()
z_ip, t_ip, _, nodos = ufl(f, c, fuerte=True)
print(f'Óptimo entero z_IP = {z_ip:.3f}')
for nombre, fuerte in (('débil', False), ('fuerte', True)):
    z_lp, _, ncons, _ = ufl(f, c, fuerte=fuerte, relajada=True)
    print(f'Relajación {nombre:7s}: z_LP = {z_lp:8.3f} · gap = {100*(z_ip - z_lp)/z_ip:5.1f} % · {ncons} restricciones')

**Qué observar.** La formulación fuerte tiene más restricciones, pero su cota inferior está mucho más cerca del óptimo (gap pequeño o nulo). La débil deja abrir «un poco» cada planta ($y_i = \sum_j x_{ij}/n$), y eso abarata artificialmente la relajación.

✏️ **Ejercicio 1.** Cambia la semilla y el tamaño (`instancia_ufl(m=12, n=40, semilla=7)`) y compara también los **nodos** de branch-and-bound de SCIP con cada formulación (`ufl(...)[3]`). ¿Se mantiene la conclusión?

In [ ]:
#@title Solución del ejercicio 1 { display-mode: "form" }
f2, c2 = instancia_ufl(m=12, n=40, semilla=7)
for nombre, fuerte in (('débil', False), ('fuerte', True)):
    z, t, _, nodos = ufl(f2, c2, fuerte=fuerte)
    zl = ufl(f2, c2, fuerte=fuerte, relajada=True)[0]
    print(f'{nombre:7s}: z_IP={z:.3f}  z_LP={zl:.3f}  nodos={nodos}  t={t:.2f}s')

## 2. Big-M ajustada

Con capacidades, la restricción de enlace es $\sum_j d_j x_{ij} \le M y_i$. Cualquier $M$ suficientemente grande es **correcta**, pero cuanto mayor es $M$, más débil es la relajación. La más pequeña válida es $M_i = \min(Q_i, \sum_j d_j)$: la planta no puede servir más que su capacidad ni más que la demanda total.

In [ ]:
def cfl_lp(M_factor, semilla=3, m=6, n=15):
    rnd = random.Random(semilla)
    d = [rnd.randint(5, 20) for _ in range(n)]
    Q = [rnd.randint(40, 90) for _ in range(m)]
    f = [rnd.randint(30, 60) for _ in range(m)]
    c = [[rnd.randint(1, 15) for _ in range(n)] for _ in range(m)]
    def resuelve(relajada, Ms):
        s = pywraplp.Solver.CreateSolver('GLOP' if relajada else 'SCIP')
        y = [s.NumVar(0, 1, '') if relajada else s.BoolVar('') for _ in range(m)]
        x = [[s.NumVar(0, 1, '') for _ in range(n)] for _ in range(m)]
        for j in range(n): s.Add(sum(x[i][j] for i in range(m)) == 1)
        for i in range(m):
            s.Add(sum(d[j] * x[i][j] for j in range(n)) <= Q[i])            # capacidad
            s.Add(sum(d[j] * x[i][j] for j in range(n)) <= Ms[i] * y[i])    # enlace con big-M
        s.Minimize(sum(f[i]*y[i] for i in range(m)) + sum(c[i][j]*d[j]*x[i][j] for i in range(m) for j in range(n)))
        s.Solve(); return s.Objective().Value()
    M_aj = [min(Q[i], sum(d)) for i in range(m)]
    z_ip = resuelve(False, M_aj)
    return z_ip, resuelve(True, [M_factor * Mi for Mi in M_aj])

for k in (1, 2, 10, 100):
    z_ip, z_lp = cfl_lp(k)
    print(f'M = {k:>3} × M_ajustada → z_LP = {z_lp:9.2f} · gap = {100*(z_ip-z_lp)/z_ip:5.1f} %')

## 3. Desigualdades de cubierta en la mochila

En $\max \sum v_i x_i$ s.a. $\sum w_i x_i \le C$, un conjunto $K$ con $\sum_{i\in K} w_i > C$ es una **cubierta**: no caben todos, así que $\sum_{i\in K} x_i \le |K|-1$ es válida. No elimina ninguna solución entera, pero sí puntos fraccionarios de la relajación.

In [ ]:
v = [10, 13, 7, 8, 6]; w = [5, 7, 4, 5, 3]; C = 12

def mochila(cubiertas=(), relajada=True):
    s = pywraplp.Solver.CreateSolver('GLOP' if relajada else 'SCIP')
    x = [s.NumVar(0, 1, f'x{i}') if relajada else s.BoolVar(f'x{i}') for i in range(len(v))]
    s.Add(sum(w[i] * x[i] for i in range(len(v))) <= C)
    for K in cubiertas: s.Add(sum(x[i] for i in K) <= len(K) - 1)
    s.Maximize(sum(v[i] * x[i] for i in range(len(v))))
    s.Solve(); return s.Objective().Value(), [round(xi.solution_value(), 3) for xi in x]

print('Óptimo entero      :', mochila(relajada=False))
print('Relajación         :', mochila())
print('Con cubierta {0,1} :', mochila(cubiertas=[(0, 1)]))

✏️ **Ejercicio 2.** Encuentra a mano **todas las cubiertas minimales** de esta mochila (quitar cualquier elemento deja de ser cubierta), añádelas y mide cuánto se cierra el gap. Comprueba tu lista con la celda de solución.

In [ ]:
#@title Solución del ejercicio 2 { display-mode: "form" }
from itertools import combinations
n = len(w)
cub = [K for r in range(2, n + 1) for K in combinations(range(n), r) if sum(w[i] for i in K) > C]
minimales = [K for K in cub if all(sum(w[i] for i in K) - w[k] <= C for k in K)]
print('Cubiertas minimales:', minimales)
z_ip = mochila(relajada=False)[0]; z0 = mochila()[0]; z1 = mochila(cubiertas=minimales)[0]
print(f'z_IP={z_ip}  z_LP={z0:.3f}  z_LP+cubiertas={z1:.3f}  → gap cerrado: {100*(z0-z1)/(z0-z_ip):.0f} %')

## 🎯 Tu instancia personal y tu entrega

1. En OptimizIA, ve a **Cuadernos → Tu instancia personal** y pulsa **Copiar**.
2. Sustituye la línea `INSTANCIA = None` de la celda siguiente por la que has copiado y ejecuta las dos celdas.
3. Copia la línea `ENTREGA = {…}` que aparece y pégala en OptimizIA: la app recalcula tu instancia con su propio solver y valida tu resultado.

> ✏️ Antes de ejecutar, **predice**: ¿qué gap tendrá la formulación débil en tu instancia? ¿Y la fuerte?

In [ ]:
INSTANCIA = None   # ← sustituye esta línea por la que copias de OptimizIA

In [ ]:
import json
if INSTANCIA:
    fI, cI = INSTANCIA['f'], INSTANCIA['c']
    res = {'z_ip': ufl(fI, cI, fuerte=True)[0], 'z_lp_debil': ufl(fI, cI, fuerte=False, relajada=True)[0], 'z_lp_fuerte': ufl(fI, cI, fuerte=True, relajada=True)[0]}
    print(f"gap débil = {100*(res['z_ip']-res['z_lp_debil'])/res['z_ip']:.1f} % · gap fuerte = {100*(res['z_ip']-res['z_lp_fuerte'])/res['z_ip']:.1f} %")
    print('ENTREGA =', json.dumps({'app': 'optimizia', 'cuaderno': 1, 'semilla': INSTANCIA['semilla'], 'res': res}))
else:
    print('Pega primero tu INSTANCIA en la celda anterior.')

## Para llevarte
- La calidad de una formulación entera se mide por su **relajación lineal**: cuanto más cerca de la envolvente convexa, mejor.
- Desagregar restricciones, ajustar las big-M y añadir desigualdades válidas (cubiertas, cortes) son formas de **reforzar** sin cambiar las soluciones enteras.
- Bibliografía de la guía: Nemhauser y Wolsey, *Integer and Combinatorial Optimization* (Wiley).